In [ ]:
import os
import gc
import pandas as pd
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

import anndata as ad
import numpy as np
import squidpy as sq
import yaml
import sys
import h5py
import logging
import scanpy as sc
from os.path import join
import scipy.io as sio
import scipy.sparse as sps
from sklearn.cluster import KMeans
import gzip
from scipy.io import mmread
from pathlib import Path, PurePath
from sklearn.metrics import adjusted_rand_score
from annoy import AnnoyIndex
import itertools
from sklearn.preprocessing import normalize

import metrics

# Lymph

In [ ]:
def load_data(_dir):
    feat_names = pd.read_csv(join(_dir, 'features.tsv.gz'), compression='gzip', sep='\t', header=None)
    barcodes   = pd.read_csv(join(_dir, 'barcodes.tsv.gz'), compression='gzip', sep='\t', header=None)

    with gzip.open(join(_dir, 'matrix.mtx.gz'), 'rb') as gzipped_file:
        mat = mmread(gzipped_file)

    ad = sc.AnnData(sps.csr_matrix(mat.T))
    ad.obs_names = barcodes[0].values
    ad.var_names = feat_names[1].values
    ad.var['id'] = feat_names[0].values
    ad.var['type'] = feat_names[2].values
    return ad

import json
import copy
from matplotlib.image import imread
def load_spatial(path, adata, library_id='0'):
    tissue_positions_file = join(path, "tissue_positions.csv")
    files = dict(
        tissue_positions_file=tissue_positions_file,
        scalefactors_json_file=join(path, "scalefactors_json.json"),
        hires_image=join(path, "tissue_hires_image.png"),
        lowres_image=join(path, "tissue_lowres_image.png"),
    )
    
    adata.uns["spatial"] = dict()
    adata.uns["spatial"][library_id] = dict()
    adata.uns["spatial"][library_id]["images"] = dict()
    for res in ["hires", "lowres"]:
        try:
            adata.uns["spatial"][library_id]["images"][res] = imread(
                str(files[f"{res}_image"])
            )
        except Exception:
            raise OSError(f"Could not find '{res}_image'")

    # read json scalefactors
    adata.uns["spatial"][library_id]["scalefactors"] = json.loads(
        Path(files["scalefactors_json_file"]).read_bytes()
    )

    # read coordinates
    positions = pd.read_csv(
        files["tissue_positions_file"],
        header=0 if Path(tissue_positions_file).name == "tissue_positions.csv" else None,
        index_col=0,
    )
    positions.columns = [
        "in_tissue",
        "array_row",
        "array_col",
        "pxl_col_in_fullres",
        "pxl_row_in_fullres",
    ]
    adata.obs = adata.obs.join(positions, how="left")
    adata.obsm["spatial"] = adata.obs[
        ["pxl_row_in_fullres", "pxl_col_in_fullres"]
    ].to_numpy()
   
    adata.obs.drop(
        columns=["pxl_row_in_fullres", "pxl_col_in_fullres"],
        inplace=True,
    )

In [ ]:
data_dir = '../../data/raw/Lymph_node/LN-2024-new/outs'

ad3 = load_data(join(data_dir, 'filtered_feature_bc_matrix'))
ad3_rna = ad3[:, ad3.var['type']=='Gene Expression'].copy()
ad3_adt = ad3[:, ad3.var['type']=='Antibody Capture'].copy()
load_spatial(join(data_dir, 'spatial'), ad3_rna)
load_spatial(join(data_dir, 'spatial'), ad3_adt)

ad3_rna.obs['src'] = ad3_adt.obs['src'] = ['s3']*ad3_rna.n_obs
ad3_rna.obs_names = [f's3-{x}' for x in ad3_rna.obs_names]
ad3_adt.obs_names = [f's3-{x}' for x in ad3_adt.obs_names]

ad3_rna.var_names_make_unique()
ad3_adt.var_names_make_unique()

data_dir = '../../data/raw/Lymph_node/lymp_tonsil_ramen'

ad_a1_rna = sc.read_h5ad(join(data_dir, 'lymph_A1/adata_RNA.h5ad'))
ad_a1_adt = sc.read_h5ad(join(data_dir, 'lymph_A1/adata_ADT.h5ad'))
meta1 = pd.read_csv(join(data_dir, 'lymph_A1/A1_LN_cloupe_Kwoh.csv'), index_col=0) 
ad_a1_rna.obs['lab'] = meta1.loc[ad_a1_rna.obs_names, 'manual'].to_list()
ad_a1_adt.obs['lab'] = meta1.loc[ad_a1_adt.obs_names, 'manual'].to_list()
ad_a1_rna.obs['src'] = ad_a1_adt.obs['src'] = ['s1'] * ad_a1_rna.n_obs
ad_a1_rna.obs_names = [f's1-{x}' for x in ad_a1_rna.obs_names]
ad_a1_adt.obs_names = [f's1-{x}' for x in ad_a1_adt.obs_names]
ad_a1_rna.var_names_make_unique()
ad_a1_adt.var_names_make_unique()

ad_d1_rna = sc.read_h5ad(join(data_dir, 'lymph_D1/adata_RNA.h5ad'))
ad_d1_adt = sc.read_h5ad(join(data_dir, 'lymph_D1/adata_ADT.h5ad'))
meta2 = pd.read_csv(join(data_dir, 'lymph_D1/D1_LN_cloupe_Kwoh.csv'), index_col=0) 
ad_d1_rna.obs['lab'] = meta2.loc[ad_d1_rna.obs_names, 'manual'].to_list()
ad_d1_adt.obs['lab'] = meta2.loc[ad_d1_adt.obs_names, 'manual'].to_list()
ad_d1_rna.obs['src'] = ad_d1_adt.obs['src'] = ['s2'] * ad_d1_rna.n_obs
ad_d1_rna.obs_names = [f's2-{x}' for x in ad_d1_rna.obs_names]
ad_d1_adt.obs_names = [f's2-{x}' for x in ad_d1_adt.obs_names]
ad_d1_rna.var_names_make_unique()
ad_d1_adt.var_names_make_unique()

## unify feature names
shared_gene = ad_a1_rna.var_names.intersection(ad_d1_rna.var_names).intersection(ad3_rna.var_names)
shared_prot = ad_a1_adt.var_names.intersection(ad_d1_adt.var_names).intersection(ad3_adt.var_names)

ad_a1_rna, ad_d1_rna, ad3_rna = ad_a1_rna[:, shared_gene].copy(), ad_d1_rna[:, shared_gene].copy(), ad3_rna[:, shared_gene].copy()
ad_a1_adt, ad_d1_adt, ad3_adt = ad_a1_adt[:, shared_prot].copy(), ad_d1_adt[:, shared_prot].copy(), ad3_adt[:, shared_prot].copy()

In [ ]:
ad_rna_all = sc.concat([ad_a1_rna, ad_d1_rna, ad3_rna])
ad_adt_all = sc.concat([ad_a1_adt, ad_d1_adt, ad3_adt])

sc.pp.highly_variable_genes(ad_rna_all, batch_key="src", flavor="seurat_v3", n_top_genes=5000)

ad_a1_rna = ad_a1_rna[:, ad_rna_all.var.query('highly_variable').index].copy()
ad_d1_rna = ad_d1_rna[:, ad_rna_all.var.query('highly_variable').index].copy()
ad3_rna = ad3_rna[:, ad_rna_all.var.query('highly_variable').index].copy()

In [ ]:
RNA_ADS = [ad_a1_rna, ad_d1_rna, ad3_rna]
ADT_ADS = [ad_a1_adt, ad_d1_adt, ad3_adt]
mod_dict = {'rna': RNA_ADS, 'adt':ADT_ADS}

In [ ]:
out_dir = '../../results/imputations'
methods = ['SpaMosaic', 'TotalVI', 'BABEL', 'MIDAS']
cv_res = []

for mi in methods:
    print(mi)
    for i in range(3):
        gt_ad = ADT_ADS[i]
        gt_X  = gt_ad.X.A if sps.issparse(gt_ad.X) else gt_ad.X.copy()
        # if mi == 'TotalVI':
        #     gt_X = np.ceil(gt_X/1000)
        pr_ad = sc.read_h5ad(join(out_dir, mi, f'Lymph_imputation/cv{i}_imputedADT.h5ad'))
        pr_ad = pr_ad[:, gt_ad.var_names].copy()
        pr_X  = pr_ad.X.A if sps.issparse(pr_ad.X) else pr_ad.X.copy()
        print(pr_X.shape, gt_X.shape)

        ## all protein
        pcc_cell, pcc_peak = metrics.PCCs(gt_X, pr_X)
        pcc_cell, pcc_peak = np.nanmean(pcc_cell), np.nanmean(pcc_peak)
        cmd_cell = metrics.CMD(pr_X, gt_X)
        cmd_peak = metrics.CMD(pr_X.T, gt_X.T)
        cv_res.append([mi, i, pcc_cell, pcc_peak, cmd_cell, cmd_peak])


In [ ]:
score_dir = '../../results/scores/Lymph_imputation'
os.makedirs(score_dir, exist_ok=True)

df_cv_res = pd.DataFrame(cv_res, columns=['method', 'cv-fold', 'PCC_cell', 'PCC_adt', 'CMD_cell', 'CMD_adt'])
df_cv_res.to_csv(f'{score_dir}/df_imputation.csv')

# Tonsil

In [ ]:
data_dir = '../../data/raw/Tonsil/TN-2024-new/outs'

ad3 = load_data(join(data_dir, 'filtered_feature_bc_matrix'))
ad3_rna = ad3[:, ad3.var['type']=='Gene Expression'].copy()
ad3_adt = ad3[:, ad3.var['type']=='Antibody Capture'].copy()
load_spatial(join(data_dir, 'spatial'), ad3_rna)
load_spatial(join(data_dir, 'spatial'), ad3_adt)

ad3_rna.obs['src'] = ad3_adt.obs['src'] = ['s3']*ad3_rna.n_obs
ad3_rna.obs_names = [f's3-{x}' for x in ad3_rna.obs_names]
ad3_adt.obs_names = [f's3-{x}' for x in ad3_adt.obs_names]

ad3_rna.var_names_make_unique()
ad3_adt.var_names_make_unique()

data_dir = '../../data/raw/Tonsil/lymp_tonsil_ramen'

ad_a1_rna = sc.read_h5ad(join(data_dir, 'tonsil_A1/adata_RNA.h5ad'))
ad_a1_adt = sc.read_h5ad(join(data_dir, 'tonsil_A1/adata_ADT.h5ad'))
ad_a1_rna.var_names_make_unique()
ad_a1_adt.var_names_make_unique()
meta1 = pd.read_csv(join(data_dir, 'tonsil_A1/A1_TNSL_Kwoh.csv'), index_col=0) 
ad_a1_rna.obs['lab'] = meta1.loc[ad_a1_rna.obs_names, 'manual'].to_list()
ad_a1_adt.obs['lab'] = meta1.loc[ad_a1_adt.obs_names, 'manual'].to_list()
ad_a1_rna.obs['src'] = ad_a1_adt.obs['src'] = ['s1'] * ad_a1_rna.n_obs
ad_a1_rna.obs_names = [f's1-{x}' for x in ad_a1_rna.obs_names]
ad_a1_adt.obs_names = [f's1-{x}' for x in ad_a1_adt.obs_names]

ad_d1_rna = sc.read_h5ad(join(data_dir, 'tonsil_D1/adata_RNA.h5ad'))
ad_d1_adt = sc.read_h5ad(join(data_dir, 'tonsil_D1/adata_ADT.h5ad'))
ad_d1_rna.var_names_make_unique()
ad_d1_adt.var_names_make_unique()
meta2 = pd.read_csv(join(data_dir, 'tonsil_D1/D1_TNSL_cloupe_Kwoh.csv'), index_col=0) 
ad_d1_rna.obs['lab'] = meta2.loc[ad_d1_rna.obs_names, 'manual'].to_list()
ad_d1_adt.obs['lab'] = meta2.loc[ad_d1_adt.obs_names, 'manual'].to_list()
ad_d1_rna.obs['src'] = ad_d1_adt.obs['src'] = ['s2'] * ad_d1_rna.n_obs
ad_d1_rna.obs_names = [f's2-{x}' for x in ad_d1_rna.obs_names]
ad_d1_adt.obs_names = [f's2-{x}' for x in ad_d1_adt.obs_names]

## unify feature names
shared_gene = ad_a1_rna.var_names.intersection(ad_d1_rna.var_names).intersection(ad3_rna.var_names)
shared_prot = ad_a1_adt.var_names.intersection(ad_d1_adt.var_names).intersection(ad3_adt.var_names)

ad_a1_rna, ad_d1_rna, ad3_rna = ad_a1_rna[:, shared_gene].copy(), ad_d1_rna[:, shared_gene].copy(), ad3_rna[:, shared_gene].copy()
ad_a1_adt, ad_d1_adt, ad3_adt = ad_a1_adt[:, shared_prot].copy(), ad_d1_adt[:, shared_prot].copy(), ad3_adt[:, shared_prot].copy()

In [ ]:
## unify feature names
shared_gene = ad_a1_rna.var_names.intersection(ad_d1_rna.var_names).intersection(ad3_rna.var_names)
shared_prot = ad_a1_adt.var_names.intersection(ad_d1_adt.var_names).intersection(ad3_adt.var_names)

ad_a1_rna, ad_d1_rna, ad3_rna = ad_a1_rna[:, shared_gene].copy(), ad_d1_rna[:, shared_gene].copy(), ad3_rna[:, shared_gene].copy()
ad_a1_adt, ad_d1_adt, ad3_adt = ad_a1_adt[:, shared_prot].copy(), ad_d1_adt[:, shared_prot].copy(), ad3_adt[:, shared_prot].copy()

In [ ]:
ad_rna_all = sc.concat([ad_a1_rna, ad_d1_rna, ad3_rna])
ad_adt_all = sc.concat([ad_a1_adt, ad_d1_adt, ad3_adt])

sc.pp.highly_variable_genes(ad_rna_all, batch_key="src", flavor="seurat_v3", n_top_genes=5000)

ad_a1_rna = ad_a1_rna[:, ad_rna_all.var.query('highly_variable').index].copy()
ad_d1_rna = ad_d1_rna[:, ad_rna_all.var.query('highly_variable').index].copy()
ad3_rna = ad3_rna[:, ad_rna_all.var.query('highly_variable').index].copy()

In [ ]:
shared_adt_names = ad_a1_adt.var_names.to_list()
for ad in [ad_a1_adt, ad_d1_adt, ad3_adt]:
    shared_adt_names = np.intersect1d(shared_adt_names, ad.var_names)
ad_a1_adt, ad_d1_adt, ad3_adt = ad_a1_adt[:, shared_adt_names].copy(), ad_d1_adt[:, shared_adt_names].copy(), ad3_adt[:, shared_adt_names].copy()

In [ ]:
RNA_ADS = [ad_a1_rna, ad_d1_rna, ad3_rna]
ADT_ADS = [ad_a1_adt, ad_d1_adt, ad3_adt]
mod_dict = {'rna': RNA_ADS, 'adt':ADT_ADS}
n_batches = 3
mod_sets = ['rna', 'adt']

In [ ]:
out_dir = '../../results/imputations'
methods = ['SpaMosaic', 'TotalVI', 'BABEL', 'MIDAS']
cv_res = []

for mi in methods:
    print(mi)
    for i in range(3):
        gt_ad = ADT_ADS[i]
        gt_X  = gt_ad.X.A if sps.issparse(gt_ad.X) else gt_ad.X.copy()
        # if mi == 'TotalVI':
        #     gt_X = np.ceil(gt_X/1000)
        pr_ad = sc.read_h5ad(join(out_dir, mi, f'Tonsil_imputation/cv{i}_imputedADT.h5ad'))
        pr_ad = pr_ad[:, gt_ad.var_names].copy()
        pr_X  = pr_ad.X.A if sps.issparse(pr_ad.X) else pr_ad.X.copy()
        print(pr_X.shape, gt_X.shape)

        ## all protein
        pcc_cell, pcc_peak = metrics.PCCs(gt_X, pr_X)
        pcc_cell, pcc_peak = np.nanmean(pcc_cell), np.nanmean(pcc_peak)
        cmd_cell = metrics.CMD(pr_X, gt_X)
        cmd_peak = metrics.CMD(pr_X.T, gt_X.T)
        cv_res.append([mi, i, pcc_cell, pcc_peak, cmd_cell, cmd_peak])


In [ ]:
score_dir = '../../results/scores/Tonsil_imputation'
os.makedirs(score_dir, exist_ok=True)

df_cv_res = pd.DataFrame(cv_res, columns=['method', 'cv-fold', 'PCC_cell', 'PCC_adt', 'CMD_cell', 'CMD_adt'])
df_cv_res.to_csv(f'{score_dir}/df_imputation.csv')